# Bans audit — is the draft complete?

The site treats "a champion was present" as `pick ∪ ban`, and the crystal ball
(round 4a) ranks champions by that presence. Presence inherits every gap in the
ban data, so before printing it we check the bans themselves.

This notebook answers four questions about the pinned model:

1. Does every game that had a ban phase carry the bans its era allows?
2. Can the same champion appear twice in one draft — banned twice, or banned and
   picked?
3. Is any ban impossible on its face — an unknown champion, a slot that does not
   exist, a champion banned before it was released?
4. Where the data is short, does the source *say* it is short, or does it lie by
   omission?

It is read-only: it does not write to `data/` and nothing here feeds
`public/data/*.json`, which only `pipeline/build.py` produces.

## Running it

```bash
cd <repo root>
uv run --project pipeline --with jupyterlab jupyter lab --no-browser --port 8888
```

The project's DuckDB is pinned (`duckdb==2.0.0`), and `--with` installs Jupyter
without touching `pyproject.toml`. The outputs are committed on purpose, so the
notebook reads as a record on GitHub; regenerate them with

```bash
uv run --project pipeline --with jupyterlab \
  jupyter nbconvert --to notebook --execute --inplace notebooks/bans-audit.ipynb
```

In [1]:
import os, pathlib, json

# The semantic SQL reads data/model/*.csv with relative paths, so the kernel has to
# sit in the repository root whether it was started there or in notebooks/.
ROOT = pathlib.Path.cwd()
for candidate in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (candidate / "data" / "model" / "fact_draft_slot.csv").exists():
        os.chdir(candidate)
        break
print("cwd:", pathlib.Path.cwd())

import duckdb
con = duckdb.connect()
for sql_file in sorted(pathlib.Path("pipeline/sql").glob("0[1-4]_*.sql")):
    con.execute(sql_file.read_text())
print("views loaded:", ", ".join(sorted(r[0] for r in con.execute(
    "select table_name from information_schema.tables where table_type='VIEW'").fetchall())))

cwd: /workspace


views loaded: ban_rows, competition_of, corpus_totals, dim_champion, dim_date, dim_event, dim_game, dim_match, dim_patch, dim_player, dim_team, era_summary, event_summary, fact_draft_slot, fact_game_side, fact_player_game, fact_player_honour, game_facts, game_rows, match_rows, patch_rows, pick_rows, player_rows, side_rows


## 1. What is in the ban table

28,306 ban rows. The interesting column is not the total: it is how many bans each
game has, beside the coverage marker the source puts on that game. A game that says
`complete` and carries four bans would be the failure this audit is looking for.

In [2]:
con.sql("""
with corpus as (select count(*) as games from dim_game where game_key <> 0)
select count(*)                                    as ban_rows,
       count(distinct game_key)                    as games_with_a_ban,
       (select games from corpus)                  as games_in_corpus,
       (select games from corpus) - count(distinct game_key) as games_without_a_ban,
       count(distinct champion_key)                as champions_banned
from fact_draft_slot
where phase = 'ban'
""").show()

┌──────────┬──────────────────┬─────────────────┬─────────────────────┬──────────────────┐
│ ban_rows │ games_with_a_ban │ games_in_corpus │ games_without_a_ban │ champions_banned │
│  int64   │      int64       │      int64      │        int64        │      int64       │
├──────────┼──────────────────┼─────────────────┼─────────────────────┼──────────────────┤
│    28306 │             3088 │            3091 │                   3 │              167 │
└──────────┴──────────────────┴─────────────────┴─────────────────────┴──────────────────┘



In [3]:
con.sql("""
with bans as (select game_key, count(*) as n from fact_draft_slot where phase='ban' group by 1),
     picks as (select game_key, count(*) as n from fact_draft_slot where phase='pick' group by 1)
select b.n as bans_in_game, g.draft_coverage, g.draft_fidelity,
       count(*) as games, min(p.n) as picks_min, max(p.n) as picks_max
from bans b
join dim_game g using (game_key)
join picks p using (game_key)
group by 1, 2, 3
order by 1, 2
""").show(max_rows=40)

┌──────────────┬────────────────┬────────────────┬───────┬───────────┬───────────┐
│ bans_in_game │ draft_coverage │ draft_fidelity │ games │ picks_min │ picks_max │
│    int64     │    varchar     │    varchar     │ int64 │   int64   │   int64   │
├──────────────┼────────────────┼────────────────┼───────┼───────────┼───────────┤
│            5 │ partial        │ unordered      │     7 │        10 │        10 │
│            6 │ partial        │ unordered      │   631 │         8 │        10 │
│            9 │ partial        │ ordered        │    15 │        10 │        10 │
│           10 │ complete       │ ordered        │  2435 │        10 │        10 │
└──────────────┴────────────────┴────────────────┴───────┴───────────┴───────────┘



The partition is exact: **every game whose bans are marked `complete` carries ten**
and **no `partial` game carries ten**. There are also three games with no bans at
all, and they are `complete` too — the 2020 Mid-Season Cup blind-pick games, where
there was no ban phase to record.

One row deserves a second look: among the six-ban games, `picks_min` is 8. That is
the IEM Season 7 game of 2013, with eight pick slots for ten players. Its bans are
complete for its era; its picks are not, and it is marked `partial` for that reason.

In [4]:
con.sql("""
select g.game_name, substr(g.date_time_utc, 1, 10) as played, g.draft_coverage,
       (select count(*) from fact_draft_slot s where s.game_key = g.game_key and s.phase='pick') as picks,
       (select count(*) from fact_draft_slot s where s.game_key = g.game_key and s.phase='ban')  as bans
from dim_game g
where g.game_key <> 0
  and g.game_key not in (select distinct game_key from fact_draft_slot where phase='ban')
order by played
""").show(max_rows=10)

┌───────────────────────────────────────────────────────────────────┬────────────┬────────────────┬───────┬───────┐
│                             game_name                             │   played   │ draft_coverage │ picks │ bans  │
│                              varchar                              │  varchar   │    varchar     │ int64 │ int64 │
├───────────────────────────────────────────────────────────────────┼────────────┼────────────────┼───────┼───────┤
│ Mid-Season Cup 2020 — FunPlus Phoenix vs JD Gaming — 2020-05-30   │ 2020-05-30 │ complete       │    10 │     0 │
│ Mid-Season Cup 2020 — Gen.G vs Top Esports — 2020-05-30           │ 2020-05-30 │ complete       │    10 │     0 │
│ Mid-Season Cup 2020 — FunPlus Phoenix vs Top Esports — 2020-05-31 │ 2020-05-31 │ complete       │    10 │     0 │
└───────────────────────────────────────────────────────────────────┴────────────┴────────────────┴───────┴───────┘



## 2. Two rule regimes, not a gap

Ten bans (five per side) arrive with 2017; before that the draft was three per side.
So "six bans" is not a missing ban list, it is the rule of its era. The check that
matters is per game against its own era's mode, not against a single global number.

In [5]:
con.sql("""
with b as (
  select s.game_key, cast(substr(g.date_time_utc,1,4) as int) as year, count(*) as bans
  from fact_draft_slot s join dim_game g using (game_key)
  where s.phase = 'ban' group by 1, 2
)
select case when year < 2017 then 'pre-2017 (3 per side)' else '2017+ (5 per side)' end as era,
       count(*)                                        as games,
       count(*) filter (where bans = 6)                as six_bans,
       count(*) filter (where bans = 10)               as ten_bans,
       count(*) filter (where bans not in (6, 10))     as other
from b group by 1 order by 1
""").show()

┌───────────────────────┬───────┬──────────┬──────────┬───────┐
│          era          │ games │ six_bans │ ten_bans │ other │
│        varchar        │ int64 │  int64   │  int64   │ int64 │
├───────────────────────┼───────┼──────────┼──────────┼───────┤
│ 2017+ (5 per side)    │  2450 │        0 │     2435 │    15 │
│ pre-2017 (3 per side) │   638 │      631 │        0 │     7 │
└───────────────────────┴───────┴──────────┴──────────┴───────┘



## 3. The integrity checks

Each row is a query whose answer must be zero. Anything else is a defect, not a
sample-size caveat.

In [6]:
checks = {
  "same champion banned twice in one game":
    """select count(*) from (select game_key, champion_key from fact_draft_slot
        where phase='ban' group by 1,2 having count(*)>1)""",
  "champion banned and picked in the same game":
    """select count(*) from (select game_key, champion_key from fact_draft_slot where phase='ban'
        intersect select game_key, champion_key from fact_draft_slot where phase='pick')""",
  "champion banned by the side that also picked it":
    """select count(*) from fact_draft_slot b
        join fact_draft_slot p on p.game_key=b.game_key and p.champion_key=b.champion_key
                              and p.phase='pick' and p.side=b.side
        where b.phase='ban'""",
  "ban row with an unknown champion (champion_key = 0)":
    "select count(*) from fact_draft_slot where phase='ban' and champion_key = 0",
  "ban slot repeated within (game, side)":
    """select count(*) from (select game_key, side, slot from fact_draft_slot
        where phase='ban' and slot is not null group by 1,2,3 having count(*)>1)""",
  "ban slot outside 1..5":
    """select count(*) from fact_draft_slot where phase='ban' and slot is not null
        and slot not between 1 and 5""",
  "unordered draft row carrying a slot":
    "select count(*) from fact_draft_slot where is_ordered = 0 and slot is not null",
  "is_ordered disagreeing with dim_game.draft_fidelity":
    """select count(*) from fact_draft_slot s join dim_game g using (game_key)
        where (g.draft_fidelity = 'ordered') <> (s.is_ordered = 1)""",
  "champion banned before its release date":
    """select count(*) from fact_draft_slot s join dim_champion c using (champion_key)
        join dim_game g using (game_key)
        where s.phase='ban' and c.release_date is not null
          and cast(substr(g.date_time_utc,1,10) as date) < cast(c.release_date as date)""",
  "champion picked before its release date":
    """select count(*) from fact_draft_slot s join dim_champion c using (champion_key)
        join dim_game g using (game_key)
        where s.phase='pick' and c.release_date is not null
          and cast(substr(g.date_time_utc,1,10) as date) < cast(c.release_date as date)""",
  "game marked complete, with bans, and not ten of them":
    """select count(*) from (select s.game_key from fact_draft_slot s join dim_game g using (game_key)
        where s.phase='ban' and g.draft_coverage='complete' group by 1 having count(*) <> 10)""",
  "game marked partial with ten bans":
    """select count(*) from (select s.game_key from fact_draft_slot s join dim_game g using (game_key)
        where s.phase='ban' and g.draft_coverage='partial' group by 1 having count(*) = 10)""",
}
width = max(len(k) for k in checks)
print(f"{'check'.ljust(width)}  result")
for name, sql in checks.items():
    value = con.execute(sql).fetchall()[0][0]
    print(f"{name.ljust(width)}  {value:>6}  {'ok' if value == 0 else 'DEFECT'}")

check                                                 result


same champion banned twice in one game                     0  ok
champion banned and picked in the same game                0  ok


champion banned by the side that also picked it            0  ok
ban row with an unknown champion (champion_key = 0)        0  ok
ban slot repeated within (game, side)                      0  ok
ban slot outside 1..5                                      0  ok


unordered draft row carrying a slot                        0  ok
is_ordered disagreeing with dim_game.draft_fidelity        0  ok
champion banned before its release date                    0  ok


champion picked before its release date                    0  ok
game marked complete, with bans, and not ten of them       0  ok
game marked partial with ten bans                          0  ok


## 4. The gaps: one ban short, and the source says so

Twenty-two games carry exactly one ban fewer than their era allows. The direction is
almost always the last ban of a side, and every one of them is marked `partial` —
the gap is declared, not silent.

In [7]:
con.sql("""
with b as (
  select game_key, count(*) as bans,
         count(*) filter (where side='blue') as blue,
         count(*) filter (where side='red')  as red
  from fact_draft_slot where phase='ban' group by 1
)
select e.event_name,
       count(*)                                  as games,
       sum(case when blue < red then 1 else 0 end) as short_blue,
       sum(case when red  < blue then 1 else 0 end) as short_red,
       min(b.bans) as bans_min, max(b.bans) as bans_max
from b
join dim_game g using (game_key)
join fact_game_side f on f.game_key = b.game_key and f.side = 'blue'
join dim_event e on e.event_key = f.event_key
where b.bans in (5, 9)
group by 1 order by 1
""").show(max_rows=30)

┌──────────────────────────────────┬───────┬────────────┬───────────┬──────────┬──────────┐
│            event_name            │ games │ short_blue │ short_red │ bans_min │ bans_max │
│             varchar              │ int64 │   int128   │  int128   │  int64   │  int64   │
├──────────────────────────────────┼───────┼────────────┼───────────┼──────────┼──────────┤
│ Esports World Cup 2026           │     2 │          1 │         1 │        9 │        9 │
│ IEM Season 11 World Championship │     2 │          2 │         0 │        9 │        9 │
│ IEM Season 7 World Championship  │     1 │          1 │         0 │        5 │        5 │
│ IPL 5                            │     1 │          0 │         1 │        5 │        5 │
│ MSI 2017                         │     2 │          2 │         0 │        9 │        9 │
│ MSI 2021                         │     1 │          1 │         0 │        9 │        9 │
│ MSI 2023                         │     1 │          0 │         1 │        9 │

In [8]:
con.sql("""
with b as (
  select game_key, count(*) as bans,
         count(*) filter (where side='blue') as blue,
         count(*) filter (where side='red')  as red
  from fact_draft_slot where phase='ban' group by 1
)
select sum(case when blue < red then 1 else 0 end) as games_short_a_blue_ban,
       sum(case when red  < blue then 1 else 0 end) as games_short_a_red_ban,
       count(*) filter (where bans in (5,9))       as games_one_short,
       round(100.0 * count(*) filter (where bans in (5,9)) / count(*), 2) as percent_of_games_with_bans
from b
""").show()

┌────────────────────────┬───────────────────────┬─────────────────┬────────────────────────────┐
│ games_short_a_blue_ban │ games_short_a_red_ban │ games_one_short │ percent_of_games_with_bans │
│         int128         │        int128         │      int64      │           double           │
├────────────────────────┼───────────────────────┼─────────────────┼────────────────────────────┤
│                     14 │                     8 │              22 │                       0.71 │
└────────────────────────┴───────────────────────┴─────────────────┴────────────────────────────┘



## 5. The legal duplicates: blind pick

A champion may appear twice in one draft in exactly one situation — blind pick,
where neither side sees the other. Two of the three 2020 Mid-Season Cup blind-pick
games contain one, and those games have no bans at all.

The query below groups by `game_key` **and not by `game_name`**, and that is not a
detail: `game_name` carries the event, the two teams and the date, but not the game
number, so 522 names are shared by 1,196 games. Grouping by name turns every
best-of-three into a false duplicate — it reports 1,590 rows where the answer is
three.

In [9]:
con.sql("""
select s.game_key, g.game_name, c.champion_name, count(*) as rows
from fact_draft_slot s
join dim_game g using (game_key)
join dim_champion c using (champion_key)
where s.phase = 'pick'
group by 1, 2, 3 having count(*) > 1
order by 1, 3
""").show(max_rows=10)

┌──────────┬───────────────────────────────────────────────────────────────────┬───────────────┬───────┐
│ game_key │                             game_name                             │ champion_name │ rows  │
│  int64   │                              varchar                              │    varchar    │ int64 │
├──────────┼───────────────────────────────────────────────────────────────────┼───────────────┼───────┤
│     1594 │ Mid-Season Cup 2020 — FunPlus Phoenix vs JD Gaming — 2020-05-30   │ Varus         │     2 │
│     2060 │ Mid-Season Cup 2020 — FunPlus Phoenix vs Top Esports — 2020-05-31 │ Thresh        │     2 │
│     2060 │ Mid-Season Cup 2020 — FunPlus Phoenix vs Top Esports — 2020-05-31 │ Varus         │     2 │
└──────────┴───────────────────────────────────────────────────────────────────┴───────────────┴───────┘



## 6. The artifact the site loads, against the table

`public/data/bans.json` is generated from this table, so the counts must agree. If
they ever stop agreeing, the artifact is stale and `npm run data:check` is the thing
that should have caught it.

In [10]:
rows = json.loads(pathlib.Path("public/data/bans.json").read_text())
sql_rows = con.execute("select count(*) from fact_draft_slot where phase='ban'").fetchall()[0][0]
print(f"bans.json rows: {len(rows)}")
print(f"fact_draft_slot bans: {sql_rows}")
print("agree:", len(rows) == sql_rows)

bans.json rows: 28306
fact_draft_slot bans: 28306
agree: True


## What this licenses, and what it does not

**The error runs in the safe direction.** A ban that is missing can only make a
champion look *less* present than it was. Presence is `pick ∪ ban` counted once per
game, so the worst case is a slightly smaller numerator for one champion in 22 of
3,088 games — it never invents a presence that did not happen.

**What it does not license:** reading a champion's ban count as complete. A
champion's *ban* count in those 22 games is a floor, not a measurement, and the same
is true of the 631 pre-2017 games if the question is about bans in a 5-per-side era.

**For the crystal ball (round 4a):** presence denominators are unaffected (games in
the window), and the `limits` of any presence answer should say the window's drafts
are `partial` where they are, with the count. The three verdicts the spec defines
already have a place for that sentence; nothing here changes a number on the site
today, because the site does not publish a presence share yet.